In [2]:
from pathlib import Path
from pptx import Presentation
from docx import Document
from unstructured.partition.auto import partition
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import requests


c:\Users\zyzai\miniconda3\envs\diss_proj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from docx import Document  # ADD THIS at the top

def extract_text_from_pptx(file_path):
    try:
        prs = Presentation(file_path)
        text = [shape.text for slide in prs.slides for shape in slide.shapes if hasattr(shape, "text")]
        return "\n".join(text)
    except Exception:
        return None

def extract_text_from_docx(file_path):
    try:
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])
    except Exception:
        return None

def extract_text_with_unstructured(file_path):
    try:
        elements = partition(file_path=file_path)
        return "\n".join([str(el) for el in elements])
    except Exception:
        return None

def extract_text(file_path):
    ext = Path(file_path).suffix.lower()
    if ext == ".pptx":
        text = extract_text_from_pptx(file_path)
    elif ext == ".docx":
        text = extract_text_from_docx(file_path)
    elif ext == ".pdf":
        text = extract_text_with_unstructured(file_path)
    else:
        text = None

    return text or extract_text_with_unstructured(file_path) or ""


In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

base_path = Path("Diss_doc")  # 🔁 replace with actual path

all_chunks = []

for folder in base_path.iterdir():
    if folder.is_dir():
        for file in folder.rglob("*"):
            if file.suffix.lower() in [".pptx", ".docx", ".pdf"]:
                raw_text = extract_text(file)
                if raw_text:
                    chunks = text_splitter.split_text(raw_text)
                    for chunk in chunks:
                        all_chunks.append({
                            "content": chunk,
                            "source": f"{folder.name}/{file.name}"
                        })


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["content"] for c in all_chunks]
metas = [c["source"] for c in all_chunks]

embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.array(embeddings))


Batches: 100%|██████████| 82/82 [00:05<00:00, 16.26it/s]


In [6]:
def search(query, top_k=5):
    query_vec = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), top_k)
    
    for i, idx in enumerate(indices[0]):
        print(f"\n--- Match {i+1} ---")
        print(f"Cosine Similarity: {scores[0][i]:.4f}")
        print(f"Source: {metas[idx]}")
        print(f"Content:\n{texts[idx][:500]}...")


In [7]:
search("project discovery phase for a fintech client")



--- Match 1 ---
Cosine Similarity: 0.5954
Source: Proposals/Copy of Informa Procurement Technology Cost Optimisation - Savings Discovery v2.pptx
Content:
Key: Scope of discovery work
Proposed team & commercials 
[ ‹#› ]
We are proposing the following team structure for this initial phase, to mobilise the programme, demonstrate a repeatable approach that can be both scaled and utilised in stakeholder engagement. 
Delivery of this initial phase is estimated at £57k (excluding VAT). Our day rates are based on the current Informa-Clarasys rate card....

--- Match 2 ---
Cosine Similarity: 0.5523
Source: Proposals/Copy of Informa Procurement Technology Cost Optimisation - Savings Discovery v2.pptx
Content:
Our team will work with Procurement, Technology & Business Owners on the Discovery. Additionally, we will work in close collaboration with the wider programme team; focusing on the scope of WP2 over a seven week discovery period (start date to be confirmed). We will also provide supportin

In [8]:
def query_mistral(prompt, model="mistral:latest"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()['response']


In [9]:
def build_prompt(query, context_chunks_with_sources):
    context_text = "\n<doc>\n".join(
        f"[{src}]\n{chunk}" for chunk, src in context_chunks_with_sources
    ) + "\n</doc>"

    prompt = f"""You are a senior consultant leading the discovery phase of a client project.

You are given excerpts from past project documents. These are strictly your ONLY source of information. Do NOT use any external knowledge.

Each excerpt starts with a filename in [brackets], followed by content. Stay within the facts only. Do NOT guess, assume, generalize, or invent anything that is not explicitly written.

--- START OF CONTEXT ---

{context_text}

--- END OF CONTEXT ---

Client request:
"{query}"

Your task:

- List all clear, stated client requirements from the request (not inferred ones).
- Find similar past projects based only on visible matches in the excerpts.
- For each similar project, include:
  - The [filename]
  - A short fact-based summary of what was done
- Mention tools, industries, or outcomes only if they appear directly in the context.
- Do NOT fabricate greetings, summaries, advice, or project plans.
- Output strictly in this bullet list format:
  - [filename]: short factual statement
  - identify the technologies used in similar previous projects
  - industries involved
  - a basic timeframe
  

Respond only with the bullet list. Nothing else.
"""
    return prompt


In [10]:
def rag(query, top_k=10, min_score=0.5):
    query_vec = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), top_k)

    context_chunks_with_sources = []
    sources = []

    for i, idx in enumerate(indices[0]):
        score = scores[0][i]
        if score >= min_score:
            chunk = texts[idx]
            source = metas[idx]
            context_chunks_with_sources.append((chunk, source))
            sources.append((score, source))

    print("\n🔍 Matches Above Threshold:")
    if not context_chunks_with_sources:
        print("Not enough info")
        return

    # for i, ((chunk, src), (score, _)) in enumerate(zip(context_chunks_with_sources, sources)):
        # print(f"\n--- Match {i+1} ---")
        # print(f"Source: {src}")
        # print(f"Score: {score:.4f}")
        # print(f"Content:\n{chunk[:700]}...")  # truncate long content for readability

    prompt = build_prompt(query, context_chunks_with_sources)
    response = query_mistral(prompt)

    print("\n LLM Response:")
    print(response)



In [11]:
rag("""From:
Anita Desai
Head of Digital Services
City of Riverton Council

Subject:
Request for Support on Citizen Services Modernization Initiative

Hi team,

We’re reaching out to explore working with your consultancy on a new initiative we're planning.

The City of Riverton is looking to modernize how we deliver public services online. Our current systems are fragmented and heavily manual — for example, applying for permits, reporting local issues, and accessing social support all happen on different platforms (some even still on paper).

Our aim is to create a unified digital experience for citizens. This would include:

    A central online portal where people can log in and access multiple services

    Backend automation to reduce manual workloads for our staff

    Better tracking and analytics to understand service usage and pain points

We’re particularly keen to learn what similar projects you’ve worked on with other public sector bodies, and how we can avoid common pitfalls. We know this will be a multi-phase project, and right now we’re just trying to get a clear picture of what the discovery and planning would look like.

Looking forward to hearing how you’d approach this.

Thanks,
Anita
    """)



🔍 Matches Above Threshold:

 LLM Response:
 - Client Requirements:
  - A central online portal where people can log in and access multiple services
  - Backend automation to reduce manual workloads for staff
  - Better tracking and analytics to understand service usage and pain points

- Similar Past Projects:
  - [Proposals/Copy of Clarasys - Immediate Media Digital Transformation Mobilisation and Discovery proposal - Sept 2024.pptx]: Proposal for a digital transformation focusing on the development of a new platform, with an investment from Clarasys.
  - [Proposals/MASTER Business Architecture & Strategy Slides - Internal - Live .pptx]: Mention of assistance, self-service, communities, automated services, and co-creation which can be related to creating digital services.
  - [Proj_doc/Copy of Ideagen & Clarasys Business Architecture Partnership Proposal - Draft - Q1 24.pptx]: A business and technology transformation project focused on the introduction or transformation of a digital 